# Tutorial 03: ML/MM molecular dynamics in OpenMM

*Author: Rhyan Barrett*

FieldMACE as an `openmm.PythonForce`, with any force field for the rest. Part 1 needs the model from tutorial 01,
part 2 the model and `work/` files from tutorial 02. Details: [README](README.md).

In [ ]:
import os, subprocess, sys
import numpy as np
from IPython.display import Image, display

QUICK = os.environ.get("FIELDMACE_TUTORIAL_QUICK") == "1"   # tiny settings for automatic checks
os.environ.setdefault("OMP_NUM_THREADS", "4")

def run(cmd, env=None):
    """Run a tutorial step and show its output (stops the notebook if the step fails)."""
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                       env={**os.environ, **(env or {})})
    lines = [l for l in p.stdout.splitlines() if "Warning" not in l and "warnings.warn" not in l]
    print("\n".join(lines[-60:]))
    if p.returncode:
        raise RuntimeError(f"step failed: {cmd}")

## Part 1: ground state – NVT, then NVE

`create_fieldmace_mm_system(mm_system, qm_atoms, model, topology=..., settings=..., **prepare_kwargs)` removes
QM–QM bonded terms, zeroes the QM charges (QM–QM and QM–MM Coulomb come from FieldMACE), keeps QM–MM
Lennard-Jones, places link atoms and adds the model. `setup.json` from the training data guarantees the same
embedding.

In [ ]:
run(f"python 01_mlmm_md.py --nvt-ps {0.01 if QUICK else 2} --nve-ps {0.01 if QUICK else 1}")

In [ ]:
import matplotlib.pyplot as plt
d = np.loadtxt("results/mlmm_nve.csv", delimiter=",")
plt.figure(figsize=(5, 3))
for k, n in ((1, "E_pot"), (2, "E_kin"), (3, "E_tot")):
    plt.plot(d[:, 0], d[:, k] - d[0, k], label=n)
plt.xlabel("time (ps)"); plt.ylabel("energy - initial (kJ/mol)"); plt.legend(); plt.title("NVE"); plt.show()

## Part 2: excited states

The active state is the context parameter `fieldmace_state`; `FieldMACEStateReporter` writes all states'
energies. The script runs on S1, switches to S0 and rescales the velocities.

In [ ]:
if os.path.exists("../02_excited_states/models/ch2s_es.model"):
    run(f"python 02_multistate_md.py --ps-per-segment {0.005 if QUICK else 0.25}")
    display(Image("results/multistate.png"))
else:
    print("train the model of tutorial 02 first")

## Part 3: active learning

ML/MM MD with one model, a committee evaluates the frames, the most uncertain ones are written for labelling.

In [ ]:
run(f"python 03_active_learning.py --n-frames {4 if QUICK else 40} --every {5 if QUICK else 100} --n-select {2 if QUICK else 10}")

## Part 4: the automated loop

`fieldmace_active_learning --config al.yaml`: train a committee, ML/MM MD, select the uncertain frames, label
them with PySCF, add them, repeat. One round here; running the same command again continues.

In [ ]:
print(open("al.yaml").read())

In [ ]:
run("bash 04_active_learning_loop.sh" + ("" if QUICK else " --rounds 1"))

## Part 5: how the interface works

What `create_fieldmace_mm_system` changes in the force field, where the link atoms go, which charges each link
scheme hides, what the model sees (embedding frame), the energy bookkeeping with force groups, and a
finite-difference check of the forces through link atoms and the switching function.

In [ ]:
run("python 05_how_it_works.py")

In [ ]:
# the same by hand: one force group for FieldMACE, then its energy alone
import openmm as mm
from openmm import unit
from mace.mlmm import create_fieldmace_mm_system
from mace.mlmm.dataset import load_setup
from mace.mlmm.io import load_system, constrain_hbonds_except
from common import default_pdb
s = load_system(pdb=default_pdb(), forcefield=["amber14-all.xml", "amber14/tip3p.xml"], constraints=None, cutoff_nm=0.9)
qm, settings, prep = load_setup("../01_data_and_training/data/setup.json")
constrain_hbonds_except(s.system, s.topology, qm)
system, spec, comp = create_fieldmace_mm_system(s.system, qm, "../01_data_and_training/models/aladip_PerAtomFieldEMACE.model",
                                                topology=s.topology, settings=settings, force_group=31,
                                                model_kwargs=dict(device="cpu", subtract_e0=True), **prep)
ctx = mm.Context(system, mm.VerletIntegrator(0.0005), mm.Platform.getPlatformByName("CPU"))
ctx.setPositions(s.positions); ctx.setPeriodicBoxVectors(*s.box_vectors)
e = lambda g: ctx.getState(getEnergy=True, groups=g).getPotentialEnergy().value_in_unit(unit.kilojoule_per_mole)
print(f"FieldMACE {e({31}):.2f} kJ/mol, force field {e(set(range(31))):.2f} kJ/mol")
print("link atoms:", [(l.qm_atom, l.mm_atom, round(l.ratio, 3)) for l in spec.link_atoms], "| point charges:", comp.last["n_mm_charges"])

## Part 6: other force fields and inputs

The same ML/MM set-up from an OpenMM force-field XML, Amber prmtop, GROMACS top/gro and a serialized System, and the
QM-selection language. (The model was trained with Amber14 charges: train with the force field you run with.)

In [ ]:
run("python 06_other_inputs.py" + (" --steps 2" if QUICK else ""))

## Part 7: MD recipes

Platform, constraints and time step, NVT then NPT, reporters, energy components by force group, restarts, and what
the embedding cutoff costs.

In [ ]:
run("python 07_md_recipes.py" + (" --nvt-ps 0.01 --npt-ps 0.02 --bench-steps 2" if QUICK else ""))
display(Image("results/recipes_components.png"))